In [1]:
# 12_convergent.ipynb
# Convergent-validity check: compare the model-based counterfactual sensitivities to
# a model-free observed lever-CIL contrast within each subgroup (primary case).
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, pyreadstat, glob, os
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from scipy.optimize import milp, LinearConstraint, Bounds, linprog
from scipy.stats import spearmanr
from math import comb
BASE = Path("..") if (Path("../data")).exists() else Path(".")
DATA, TAB = BASE/"data", BASE/"results/tables"; TAB.mkdir(parents=True, exist_ok=True)
PVS=[f"PV{i}CIL" for i in range(1,6)]
def wR2(y,yh,w): yb=np.average(y,weights=w); return 1-np.sum(w*(y-yh)**2)/np.sum(w*(y-yb)**2)
def wpct(v,wt,p): m=~np.isnan(v); vs=np.sort(v[m]); ww=wt[m][np.argsort(v[m])]; return np.interp(p,np.cumsum(ww)/ww.sum(),vs)
LEV=["S_LRNINTO","S_ICTPOSS","S_GENCLASS","S_LRNSAFE"]
CTX=["NISB","S_HISEI","S_HOMLIT","S_INTNET","S_EXCOMP","S_GENEFF","S_SPECEFF","S_ICTPOSG","S_ICTFUT","S_ACMULT","S_LRNINTS","S_CTCLS","S_SPECLASS","S_ICTSTUD"]

stu,_=pyreadstat.read_sas7bdat(str(DATA/"bsgkori3.sas7bdat"))
for c in LEV+["NISB"]: stu.loc[(stu[c]>=995)|(stu[c]<=-995),c]=np.nan
stu.loc[(stu.NISB<-6)|(stu.NISB>6),"NISB"]=np.nan; stu["CIL5"]=stu[PVS].mean(1)
Z=stu[LEV].apply(lambda s:(s-s.mean())/s.std()); stu["lever"]=Z.mean(1)
w=stu.TOTWGTS.to_numpy(); x=stu.NISB.to_numpy(); ok=~np.isnan(x)
o=np.argsort(x[ok]); cw=np.cumsum(w[ok][o])/w[ok].sum(); q=np.interp([1/3,2/3],cw,x[ok][o])
stu["SES3"]=pd.Series(np.where(x<=q[0],"Low",np.where(x<=q[1],"Mid","High"))).where(ok).values
stu["SEX"]=stu.S_SEX.map({0.0:"Sex0",1.0:"Sex1"}); d=stu.dropna(subset=["SES3","SEX","lever"]).copy(); d["grp"]=d.SES3+"_"+d.SEX
def contrast(z):
    med=np.median(z.lever); hiv=z[z.lever>=med]; lov=z[z.lever<med]
    if len(hiv)<10 or len(lov)<10: return np.nan
    return np.average(hiv.CIL5,weights=hiv.TOTWGTS)-np.average(lov.CIL5,weights=lov.TOTWGTS)
obs=d.groupby("grp").apply(contrast).rename("observed_contrast").reset_index()
cf=pd.read_csv(TAB/"rev_m2_ci.csv")[["grp","dCIL"]]; M=obs.merge(cf,on="grp")
rho,p=spearmanr(M.observed_contrast,M.dCIL); M.to_csv(TAB/"c4_convergent.csv",index=False)
print(M.round(2).to_string(index=False))
print(f"all six observed contrasts positive: {(M.observed_contrast>0).all()}")
print(f"convergent validity: Spearman rho={rho:.2f} (p={p:.3f}), Pearson r={np.corrcoef(M.observed_contrast,M.dCIL)[0,1]:.2f}")


      grp  observed_contrast  dCIL
High_Sex0              14.78 16.36
High_Sex1              24.87 11.94
 Low_Sex0              27.22 19.94
 Low_Sex1              27.84 17.30
 Mid_Sex0              28.79 15.38
 Mid_Sex1              13.35 12.96
all six observed contrasts positive: True
convergent validity: Spearman rho=0.37 (p=0.468), Pearson r=0.37
